In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib


In [ ]:
DATA_PATH = os.path.join(os.path.dirname(__file__),
                         "diabetes_binary_health_indicators_BRFSS2015.csv")

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Memory before downcast: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

BINARY_COLS = [
    "Diabetes_binary", "HighBP", "HighChol", "CholCheck", "Smoker",
    "Stroke", "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"
]
ORDINAL_COLS = ["GenHlth", "Age", "Education", "Income"]
CONTINUOUS_COLS = ["BMI", "MentHlth", "PhysHlth"]

for col in BINARY_COLS + ORDINAL_COLS:
    df[col] = df[col].astype(np.int8)

for col in CONTINUOUS_COLS:
    df[col] = df[col].astype(np.float32)

print(f"  Memory after downcast:  {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")



FEATURE_COLS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income"
]

X = df[FEATURE_COLS]
y = df["Diabetes_binary"]

print(f"\nTarget distribution:")
print(y.value_counts().to_string())
print(f"   Class 1 ratio: {y.mean():.3f}")



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\nTrain/Test split:")
print(f"   Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")



scaler = StandardScaler()
X_train[CONTINUOUS_COLS] = scaler.fit_transform(X_train[CONTINUOUS_COLS])
X_test[CONTINUOUS_COLS] = scaler.transform(X_test[CONTINUOUS_COLS])

print(f"\nScaled continuous features: {CONTINUOUS_COLS}")



print(f"\nApplying SMOTE to training data...")
print(f"   Before SMOTE: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"After SMOTE:  { {0: (y_train_resampled == 0).sum(), 1: (y_train_resampled == 1).sum()} }")



print(f"\nTraining XGBClassifier...")

model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    scale_pos_weight=3,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_resampled, y_train_resampled, verbose=False)
print("Training complete.")



y_prob = model.predict_proba(X_test)[:, 1]

DECISION_THRESHOLD = 0.30
y_pred = (y_prob >= DECISION_THRESHOLD).astype(int)

print(f"\n📈 Classification Report (Test Set, threshold={DECISION_THRESHOLD}):")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))

cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:")
print(f"   TN={cm[0][0]:,}  FP={cm[0][1]:,}")
print(f"   FN={cm[1][0]:,}  TP={cm[1][1]:,}")

from sklearn.metrics import recall_score
recall_class1 = recall_score(y_test, y_pred, pos_label=1)
print(f"\nClass 1 Recall (Sensitivity): {recall_class1:.4f}")

if recall_class1 < 0.70:
    print("Recall below 0.70 threshold!")
else:
    print("Recall meets clinical threshold (≥ 0.70)")



MODEL_PATH = os.path.join(os.path.dirname(__file__), "xgb_diabetes_model.pkl")
SCALER_PATH = os.path.join(os.path.dirname(__file__), "scaler.pkl")
META_PATH = os.path.join(os.path.dirname(__file__), "model_meta.pkl")

joblib.dump(model, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
joblib.dump({
    "feature_cols": FEATURE_COLS,
    "continuous_cols": CONTINUOUS_COLS,
    "recall_class1": recall_class1,
    "decision_threshold": DECISION_THRESHOLD,
}, META_PATH)

print(f"\nSaved artifacts:")
print(f"   Model:  {MODEL_PATH}")
print(f"   Scaler: {SCALER_PATH}")
print(f"   Meta:   {META_PATH}")
print(f"\nPipeline complete.")